# Information management demo: register audit on Google Colab

A runnable example of how information is organised, checked and reported on a large infrastructure project.
It uses a **fictional** tunnelling scheme (the Marlow Valley Tunnel Scheme) and **entirely synthetic** data.

**What the notebook does**

1. Writes the whole project into Colab (rules, code, tests, documents).
2. Generates a synthetic document register with known errors planted in it.
3. Runs 11 automated quality checks, applies the safe fixes and logs them.
4. Shows the results, charts and the exceptions queue that still needs a person.
5. Runs the tests and lets you download everything as a zip.

**How to use it:** choose **Runtime > Run all**. It takes under a minute and needs no setup, accounts or files.

## Step 1. Set up the project folder

In [ ]:
!pip -q install pytest
import os
PROJECT = "/content/cde-information-management-demo"
for folder in ["config", "cde", "tests", "docs", "data", "reports"]:
    os.makedirs(f"{PROJECT}/{folder}", exist_ok=True)
%cd {PROJECT}

## Step 2. Write the project files

Each cell below writes one file. Expand a cell to read it. The code cells are the **standard** (rules and code
lists), the **generator**, the **audit**, the **fixer**, the **report builder** and the **tests**; the markdown
files are the naming standard, the procedure and the data protection note.

In [ ]:
%%writefile config/standard.json
{
  "project_code": "MVT",
  "project_name": "Marlow Valley Tunnel Scheme (fictional)",
  "originators": {
    "ARC": "Arden Civil Design",
    "KTL": "Kestrel Tunnelling",
    "FGS": "Fieldgate Surveys",
    "MVC": "Marlow Valley Constructors",
    "CLT": "Client team"
  },
  "sections": {
    "S1": "Tunnel section 1",
    "S2": "Tunnel section 2",
    "S3": "Tunnel section 3",
    "S4": "Tunnel section 4",
    "S5": "Tunnel section 5",
    "S6": "Tunnel section 6",
    "ZZ": "Project-wide"
  },
  "types": {
    "DR": "Drawing",
    "RP": "Report",
    "MS": "Method statement",
    "SP": "Specification",
    "RA": "Risk assessment",
    "CA": "Calculation",
    "LT": "Letter",
    "MI": "Meeting minutes"
  },
  "disciplines": {
    "CV": "Civil",
    "ST": "Structural",
    "GE": "Geotechnical",
    "ME": "Mechanical",
    "EL": "Electrical",
    "HS": "Health and safety",
    "PM": "Project management",
    "SU": "Survey",
    "EN": "Environmental"
  },
  "statuses": ["WIP", "SHARED", "PUBLISHED", "ARCHIVED"],
  "suitability": {
    "S0": "Work in progress (not for sharing)",
    "S1": "Suitable for coordination",
    "S2": "Suitable for information",
    "S3": "Suitable for review and comment",
    "S4": "Suitable for stage approval",
    "A": "Authorised and accepted"
  },
  "status_rules": {
    "WIP": { "suitability": ["S0"], "revision_prefix": "P" },
    "SHARED": { "suitability": ["S1", "S2", "S3"], "revision_prefix": "P" },
    "PUBLISHED": { "suitability": ["S4", "A"], "revision_prefix": "C" },
    "ARCHIVED": { "suitability": null, "revision_prefix": null }
  },
  "retention": {
    "classes": {
      "R12": "Retain 12 years after project completion (illustrative)",
      "R6": "Retain 6 years after project completion (illustrative)"
    },
    "by_type": {
      "DR": "R12",
      "RP": "R12",
      "MS": "R12",
      "SP": "R12",
      "RA": "R12",
      "CA": "R12",
      "LT": "R6",
      "MI": "R6"
    }
  },
  "classifications": ["Internal", "Restricted"]
}

In [ ]:
%%writefile cde/__init__.py
"""Information management demo: naming standard, register audit and reporting."""

In [ ]:
%%writefile cde/standard.py
"""Load the naming and metadata standard and provide filename helpers."""

from __future__ import annotations

import json
import re
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent
DEFAULT_STANDARD = ROOT / "config" / "standard.json"

REGISTER_FIELDS = [
    "record_id",
    "file_name",
    "title",
    "originator",
    "section",
    "doc_type",
    "discipline",
    "number",
    "revision",
    "status",
    "suitability",
    "author",
    "author_email",
    "created",
    "last_modified",
    "review_due",
    "content_hash",
    "contains_personal_data",
    "classification",
    "retention_class",
]

# review_due is also required, except for ARCHIVED documents (checked in audit.py).
REQUIRED_FIELDS = [f for f in REGISTER_FIELDS if f not in ("record_id", "review_due")]


def _loose(text: str) -> str:
    """Upper-case and strip everything except letters and digits."""
    return re.sub(r"[^A-Z0-9]", "", text.upper())


class Standard:
    """Wraps config/standard.json and the filename rules derived from it."""

    def __init__(self, data: dict):
        self.data = data
        self.project_code: str = data["project_code"]
        self.originators: dict = data["originators"]
        self.sections: dict = data["sections"]
        self.types: dict = data["types"]
        self.disciplines: dict = data["disciplines"]
        self.statuses: list = data["statuses"]
        self.suitability: dict = data["suitability"]
        self.status_rules: dict = data["status_rules"]
        self.retention_by_type: dict = data["retention"]["by_type"]
        self.classifications: list = data["classifications"]
        code = re.escape(self.project_code)
        self._pattern = re.compile(
            rf"^{code}-(?P<originator>[A-Z]{{3}})-(?P<section>[A-Z0-9]{{2}})"
            rf"-(?P<doc_type>[A-Z]{{2}})-(?P<discipline>[A-Z]{{2}})"
            rf"-(?P<number>\d{{4}})_(?P<revision>[PC]\d{{2}})\.(?P<ext>[a-z0-9]{{2,4}})$"
        )

    @classmethod
    def load(cls, path: Path | str = DEFAULT_STANDARD) -> "Standard":
        with open(path, encoding="utf-8") as fh:
            return cls(json.load(fh))

    # ------------------------------------------------------------------ filenames
    def parse_filename(self, name: str) -> dict | None:
        """Return the fields encoded in a file name, or None if it does not conform."""
        match = self._pattern.match(name or "")
        return match.groupdict() if match else None

    def build_filename(self, originator, section, doc_type, discipline, number, revision, ext="pdf") -> str:
        return (
            f"{self.project_code}-{originator}-{section}-{doc_type}-{discipline}"
            f"-{number}_{revision}.{ext}"
        )

    def expected_filename(self, rec: dict) -> str | None:
        """File name implied by the register metadata, or None if metadata is unusable."""
        match = re.search(r"\.([A-Za-z0-9]{2,4})$", (rec.get("file_name") or "").strip())
        ext = match.group(1).lower() if match else "pdf"
        name = self.build_filename(
            rec.get("originator", ""),
            rec.get("section", ""),
            rec.get("doc_type", ""),
            rec.get("discipline", ""),
            rec.get("number", ""),
            rec.get("revision", ""),
            ext,
        )
        return name if self.parse_filename(name) else None

    def suggest_filename_fix(self, rec: dict) -> str | None:
        """Propose a corrected file name when the problem is purely cosmetic.

        Cosmetic means the name, ignoring case and punctuation, is the expected name,
        or is the expected name with the revision missing. Anything else (extra words,
        different codes) needs a human decision, so None is returned.
        """
        expected = self.expected_filename(rec)
        if expected is None:
            return None
        name = (rec.get("file_name") or "").strip()
        stem = name.rsplit(".", 1)[0] if re.search(r"\.[A-Za-z0-9]{2,4}$", name) else name
        expected_stem = expected.rsplit(".", 1)[0]
        revision = rec.get("revision", "")
        if _loose(stem) == _loose(expected_stem) or _loose(expected_stem) == _loose(stem) + _loose(revision):
            return expected
        return None

    # ------------------------------------------------------------------ rules
    def expected_retention(self, doc_type: str) -> str | None:
        return self.retention_by_type.get(doc_type)

In [ ]:
%%writefile cde/generate.py
"""Generate a synthetic document register, optionally seeded with known errors.

Everything here is fictional. People are made up from short name lists and every
e-mail address uses the reserved example.test domain. The generator records exactly
which errors it planted (the "ground truth") so the audit can be measured against it.
"""

from __future__ import annotations

import csv
import random
from datetime import date, timedelta
from pathlib import Path

from .standard import REGISTER_FIELDS, Standard

FIRST_NAMES = ["Amira", "Ben", "Chloe", "Dev", "Elena", "Femi", "Grace", "Hamza", "Isla", "Jon", "Kira", "Liam"]
LAST_NAMES = ["Ahmed", "Brooks", "Clarke", "Dhillon", "Evans", "Fraser", "Gill", "Hughes", "Iqbal", "Jones", "Khan", "Lowe"]

TOPICS = {
    "CV": ["Shaft headwall design", "Access road drainage", "Compound layout"],
    "ST": ["Segmental lining design", "Shaft base slab check", "Connection details"],
    "GE": ["Ground investigation summary", "Settlement assessment", "Groundwater monitoring"],
    "ME": ["Ventilation plan", "Pump arrangement", "Lifting equipment register"],
    "EL": ["Temporary power layout", "Lighting design", "Earthing arrangement"],
    "HS": ["Confined space procedure", "Emergency response plan", "Permit to work process"],
    "PM": ["Progress summary", "Interface schedule", "Change request log"],
    "SU": ["Control point survey", "As-built setting out", "Alignment verification"],
    "EN": ["Noise monitoring results", "Spoil management plan", "Ecology constraints"],
}

# (check name, number of records to plant it in)
ERROR_PLAN = [
    ("filename_pattern", 5),
    ("unknown_code", 3),
    ("filename_metadata_mismatch", 4),
    ("missing_metadata", 5),
    ("status_consistency", 5),
    ("overdue_review", 6),
    ("date_error", 3),
    ("personal_data_classification", 4),
    ("retention_class", 4),
]
DUPLICATE_NUMBER_PAIRS = 3
DUPLICATE_CONTENT_PAIRS = 3


def _topic_title(rng: random.Random, std: Standard, discipline: str, section: str) -> str:
    return f"{rng.choice(TOPICS[discipline])} - {std.sections[section]}"


def _person(rng: random.Random) -> tuple[str, str]:
    first, last = rng.choice(FIRST_NAMES), rng.choice(LAST_NAMES)
    return f"{first} {last}", f"{first}.{last}@example.test".lower()


def _make_clean_records(rng: random.Random, std: Standard, n: int, as_of: date) -> list[dict]:
    records: list[dict] = []
    used_keys: set[tuple] = set()
    statuses = std.statuses
    created_start = date(2025, 9, 1)
    while len(records) < n:
        originator = rng.choice(list(std.originators))
        section = rng.choice(list(std.sections))
        doc_type = rng.choice(list(std.types))
        discipline = rng.choice(list(std.disciplines))
        number = f"{rng.randint(1, 250):04d}"
        key = (originator, section, doc_type, discipline, number)
        if key in used_keys:
            continue
        used_keys.add(key)

        status = rng.choices(statuses, weights=[20, 30, 40, 10])[0]
        rule = std.status_rules[status]
        suitability = rng.choice(rule["suitability"] or list(std.suitability))
        prefix = rule["revision_prefix"] or rng.choice("PC")
        revision = f"{prefix}{rng.randint(1, 4):02d}"
        ext = rng.choice(["pdf", "pdf", "pdf", "docx", "xlsx", "dwg"])

        created = created_start + timedelta(days=rng.randint(0, 395))
        last_modified = created + timedelta(days=rng.randint(0, 120))
        last_modified = min(last_modified, as_of - timedelta(days=1))
        review_due = "" if status == "ARCHIVED" else (as_of + timedelta(days=rng.randint(14, 150))).isoformat()

        personal = "Y" if doc_type in ("LT", "MI") or rng.random() < 0.1 else "N"
        author, email = _person(rng)

        records.append(
            {
                "record_id": f"REG-{len(records) + 1:04d}",
                "file_name": std.build_filename(originator, section, doc_type, discipline, number, revision, ext),
                "title": _topic_title(rng, std, discipline, section),
                "originator": originator,
                "section": section,
                "doc_type": doc_type,
                "discipline": discipline,
                "number": number,
                "revision": revision,
                "status": status,
                "suitability": suitability,
                "author": author,
                "author_email": email,
                "created": created.isoformat(),
                "last_modified": last_modified.isoformat(),
                "review_due": review_due,
                "content_hash": f"{rng.getrandbits(64):016x}",
                "contains_personal_data": personal,
                "classification": "Restricted" if personal == "Y" else "Internal",
                "retention_class": std.expected_retention(doc_type),
            }
        )
    return records


# --------------------------------------------------------------------------- error injection
def _swap_filename_part(rec: dict, std: Standard, **changes) -> None:
    parsed = std.parse_filename(rec["file_name"])
    parsed.update(changes)
    rec["file_name"] = std.build_filename(
        parsed["originator"], parsed["section"], parsed["doc_type"],
        parsed["discipline"], parsed["number"], parsed["revision"], parsed["ext"],
    )


def _inject(check: str, rec: dict, rng: random.Random, std: Standard, as_of: date) -> None:
    if check == "filename_pattern":
        good = rec["file_name"]
        stem, ext = good.rsplit(".", 1)
        variant = rng.choice(["lowercase", "underscores", "no_revision", "extra_word", "lowercase"])
        if variant == "lowercase":
            rec["file_name"] = f"{stem.lower()}.{ext}"
        elif variant == "underscores":
            rec["file_name"] = f"{stem.replace('-', '_')}.{ext}"
        elif variant == "no_revision":
            rec["file_name"] = f"{stem.split('_')[0]}.{ext}"
        else:
            rec["file_name"] = f"{stem} FINAL.{ext}"

    elif check == "unknown_code":
        field, bad = rng.choice(
            [("originator", "XYZ"), ("section", "S9"), ("doc_type", "ZZ"), ("discipline", "QQ")]
        )
        rec[field] = bad
        _swap_filename_part(rec, std, **{field: bad})

    elif check == "filename_metadata_mismatch":
        field = rng.choice(["section", "discipline", "originator", "revision"])
        if field == "section":
            new = rng.choice([c for c in std.sections if c != rec["section"]])
        elif field == "discipline":
            new = rng.choice([c for c in std.disciplines if c != rec["discipline"]])
        elif field == "originator":
            new = rng.choice([c for c in std.originators if c != rec["originator"]])
        else:
            current = int(rec["revision"][1:])
            new = f"{rec['revision'][0]}{(current % 4) + 1:02d}"
        _swap_filename_part(rec, std, **{field: new})

    elif check == "missing_metadata":
        field = rng.choice(["title", "author", "author_email", "review_due"])
        if field == "review_due" and rec["status"] == "ARCHIVED":
            field = "title"
        rec[field] = ""

    elif check == "status_consistency":
        rule = std.status_rules[rec["status"]]
        if rng.random() < 0.5:
            wrong = [s for s in std.suitability if s not in rule["suitability"]]
            rec["suitability"] = rng.choice(wrong)
        else:
            new_prefix = "C" if rule["revision_prefix"] == "P" else "P"
            new_rev = f"{new_prefix}{rec['revision'][1:]}"
            rec["revision"] = new_rev
            _swap_filename_part(rec, std, revision=new_rev)

    elif check == "overdue_review":
        rec["review_due"] = (as_of - timedelta(days=rng.randint(5, 90))).isoformat()

    elif check == "date_error":
        created = date.fromisoformat(rec["created"])
        rec["last_modified"] = (created - timedelta(days=rng.randint(1, 30))).isoformat()

    elif check == "personal_data_classification":
        rec["contains_personal_data"] = "Y"
        rec["classification"] = "Internal"

    elif check == "retention_class":
        correct = rec["retention_class"]
        rec["retention_class"] = rng.choice([c for c in ("R6", "R12") if c != correct])

    else:  # pragma: no cover
        raise ValueError(check)


def _needs_live(check: str) -> bool:
    """Checks that only make sense for documents that are not archived."""
    return check in ("overdue_review", "status_consistency")


def generate(
    std: Standard,
    seed: int = 42,
    n_documents: int = 120,
    as_of: date = date(2026, 11, 12),
    inject_errors: bool = True,
) -> tuple[list[dict], list[dict]]:
    """Return (records, ground_truth). ground_truth rows are {record_id, check, detail}."""
    rng = random.Random(seed)
    records = _make_clean_records(rng, std, n_documents, as_of)
    truth: list[dict] = []
    if not inject_errors:
        return records, truth

    # Choose victims without replacement so each planted error has its own record.
    pool = list(range(len(records)))
    rng.shuffle(pool)

    def take(check: str) -> int:
        for pos, idx in enumerate(pool):
            if _needs_live(check) and records[idx]["status"] == "ARCHIVED":
                continue
            return pool.pop(pos)
        raise RuntimeError("not enough records to plant errors")

    # Reserve originals for the duplicate pairs first so they stay otherwise clean.
    dup_number_src = [take("duplicate_number") for _ in range(DUPLICATE_NUMBER_PAIRS)]
    dup_content_src = [take("duplicate_content") for _ in range(DUPLICATE_CONTENT_PAIRS)]

    for check, count in ERROR_PLAN:
        for _ in range(count):
            idx = take(check)
            rec = records[idx]
            _inject(check, rec, rng, std, as_of)
            truth.append({"record_id": rec["record_id"], "check": check, "detail": "planted"})

    next_id = len(records) + 1

    # Same document number and revision, different content: a numbering clash.
    for idx in dup_number_src:
        src = records[idx]
        clone = dict(src)
        clone["record_id"] = f"REG-{next_id:04d}"
        next_id += 1
        clone["title"] = src["title"] + " (updated)"
        clone["author"], clone["author_email"] = _person(rng)
        clone["content_hash"] = f"{rng.getrandbits(64):016x}"
        records.append(clone)
        for rid in (src["record_id"], clone["record_id"]):
            truth.append({"record_id": rid, "check": "duplicate_number", "detail": "planted pair"})

    # Same content filed twice under two different (valid) document numbers.
    used = {(r["originator"], r["section"], r["doc_type"], r["discipline"], r["number"]) for r in records}
    for idx in dup_content_src:
        src = records[idx]
        clone = dict(src)
        clone["record_id"] = f"REG-{next_id:04d}"
        next_id += 1
        while True:
            number = f"{rng.randint(251, 999):04d}"
            key = (src["originator"], src["section"], src["doc_type"], src["discipline"], number)
            if key not in used:
                used.add(key)
                break
        clone["number"] = number
        clone["file_name"] = std.build_filename(
            src["originator"], src["section"], src["doc_type"], src["discipline"], number,
            src["revision"], src["file_name"].rsplit(".", 1)[1],
        )
        records.append(clone)
        for rid in (src["record_id"], clone["record_id"]):
            truth.append({"record_id": rid, "check": "duplicate_content", "detail": "planted pair"})

    return records, truth


# --------------------------------------------------------------------------- csv helpers
def write_register(records: list[dict], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=REGISTER_FIELDS)
        writer.writeheader()
        writer.writerows(records)


def read_register(path: Path) -> list[dict]:
    with open(path, newline="", encoding="utf-8") as fh:
        return list(csv.DictReader(fh))


def write_rows(rows: list[dict], path: Path, fieldnames: list[str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

In [ ]:
%%writefile cde/audit.py
"""Quality checks for the document register.

Each check returns findings of the form:
    record_id, file_name, check, severity, field, message, suggested_action, auto_fixable
The audit only looks at the register itself; it never sees the generator's ground truth.
"""

from __future__ import annotations

import re
from collections import defaultdict
from datetime import date

from .standard import REQUIRED_FIELDS, Standard

FINDING_FIELDS = [
    "record_id",
    "file_name",
    "check",
    "severity",
    "field",
    "message",
    "suggested_action",
    "auto_fixable",
]

CHECKS = {
    "filename_pattern": ("Medium", "File name does not follow the naming standard"),
    "filename_metadata_mismatch": ("High", "File name disagrees with the register metadata"),
    "unknown_code": ("Medium", "A code is not in the approved code lists"),
    "missing_metadata": ("Medium", "Required metadata is blank"),
    "status_consistency": ("High", "Status, suitability and revision do not agree"),
    "duplicate_number": ("High", "Two records share a document number and revision"),
    "duplicate_content": ("Medium", "Identical content filed under different numbers"),
    "overdue_review": ("Medium", "Review date has passed"),
    "date_error": ("Low", "Dates are invalid or out of order"),
    "personal_data_classification": ("High", "Personal data is not classed as Restricted"),
    "retention_class": ("Low", "Retention class does not match the document type"),
}

_REVISION_RE = re.compile(r"^[PC]\d{2}$")


def _parse_date(value: str) -> date | None:
    try:
        return date.fromisoformat(value)
    except (TypeError, ValueError):
        return None


def audit(records: list[dict], std: Standard, as_of: date) -> list[dict]:
    findings: list[dict] = []

    def add(rec, check, field, message, action, auto=False):
        findings.append(
            {
                "record_id": rec["record_id"],
                "file_name": rec.get("file_name", ""),
                "check": check,
                "severity": CHECKS[check][0],
                "field": field,
                "message": message,
                "suggested_action": action,
                "auto_fixable": "Y" if auto else "N",
            }
        )

    for rec in records:
        status = rec.get("status", "")

        # 1. Required metadata present
        missing = [f for f in REQUIRED_FIELDS if not (rec.get(f) or "").strip()]
        if status != "ARCHIVED" and not (rec.get("review_due") or "").strip():
            missing.append("review_due")
        if missing:
            add(rec, "missing_metadata", ";".join(missing),
                f"Missing required field(s): {', '.join(missing)}",
                "Ask the author or owner to complete the metadata")

        # 2. File name follows the standard
        parsed = std.parse_filename(rec.get("file_name", ""))
        if parsed is None:
            fix = std.suggest_filename_fix(rec)
            if fix:
                add(rec, "filename_pattern", "file_name",
                    f"'{rec.get('file_name', '')}' does not match the pattern; expected '{fix}'",
                    "Rename to the expected file name", auto=True)
            else:
                add(rec, "filename_pattern", "file_name",
                    f"'{rec.get('file_name', '')}' does not match the naming pattern",
                    "Author to confirm the correct document number and rename")
        else:
            # 3. File name agrees with metadata
            differing = [f for f in ("originator", "section", "doc_type", "discipline", "number", "revision")
                         if parsed[f] != rec.get(f, "")]
            if differing:
                add(rec, "filename_metadata_mismatch", ";".join(differing),
                    "File name and metadata disagree on: " + ", ".join(differing),
                    "Check which is correct, then correct the file name or the register entry")

        # 4. Codes are on the approved lists
        unknown = []
        for field, valid in (("originator", std.originators), ("section", std.sections),
                             ("doc_type", std.types), ("discipline", std.disciplines),
                             ("suitability", std.suitability)):
            value = rec.get(field, "")
            if value and value not in valid:
                unknown.append(f"{field}={value}")
        if rec.get("status") and rec["status"] not in std.statuses:
            unknown.append(f"status={rec['status']}")
        if rec.get("revision") and not _REVISION_RE.match(rec["revision"]):
            unknown.append(f"revision={rec['revision']}")
        if unknown:
            add(rec, "unknown_code", ";".join(u.split("=")[0] for u in unknown),
                "Code not in the approved lists: " + ", ".join(unknown),
                "Information Manager to confirm whether a new code is needed or the entry is wrong")

        # 5. Status, suitability and revision agree
        rule = std.status_rules.get(status)
        if rule:
            problems = []
            if rule["suitability"] and rec.get("suitability") in std.suitability \
                    and rec["suitability"] not in rule["suitability"]:
                problems.append(f"suitability {rec['suitability']} not allowed for {status}")
            prefix = (rec.get("revision") or "")[:1]
            if rule["revision_prefix"] and _REVISION_RE.match(rec.get("revision", "")) \
                    and prefix != rule["revision_prefix"]:
                problems.append(f"revision {rec['revision']} should start with {rule['revision_prefix']} for {status}")
            if problems:
                add(rec, "status_consistency", "status;suitability;revision", "; ".join(problems),
                    "Author to correct the status, suitability or revision")

        # 8. Review overdue
        due = _parse_date(rec.get("review_due", ""))
        if status != "ARCHIVED" and due and due < as_of:
            add(rec, "overdue_review", "review_due",
                f"Review was due {due.isoformat()} ({(as_of - due).days} days overdue)",
                "Owner to review the document and set a new review date")

        # 9. Dates valid and in order
        created, modified = _parse_date(rec.get("created", "")), _parse_date(rec.get("last_modified", ""))
        date_problems = []
        for field in ("created", "last_modified", "review_due"):
            if (rec.get(field) or "").strip() and _parse_date(rec[field]) is None:
                date_problems.append(f"{field} is not a valid ISO date")
        if created and modified and modified < created:
            date_problems.append("last_modified is earlier than created")
        if created and created > as_of:
            date_problems.append("created is in the future")
        if date_problems:
            add(rec, "date_error", "created;last_modified", "; ".join(date_problems),
                "Owner to correct the dates")

        # 10. Personal data must be Restricted
        if rec.get("contains_personal_data") == "Y" and rec.get("classification") != "Restricted":
            add(rec, "personal_data_classification", "classification",
                f"Flagged as containing personal data but classified '{rec.get('classification', '')}'",
                "Set classification to Restricted", auto=True)

        # 11. Retention class matches document type
        expected = std.expected_retention(rec.get("doc_type", ""))
        if expected and rec.get("retention_class") != expected:
            add(rec, "retention_class", "retention_class",
                f"Retention class is '{rec.get('retention_class', '')}' but {rec['doc_type']} documents use {expected}",
                f"Set retention class to {expected}", auto=True)

    # 6. Duplicate document number and revision
    groups = defaultdict(list)
    for rec in records:
        key = tuple(rec.get(f, "") for f in ("originator", "section", "doc_type", "discipline", "number", "revision"))
        if all(key):
            groups[key].append(rec)
    for members in groups.values():
        if len(members) > 1:
            ids = ", ".join(m["record_id"] for m in members)
            for rec in members:
                add(rec, "duplicate_number", "number;revision",
                    f"Same number and revision used by {ids}",
                    "Information Manager to reissue one number and reconcile the content")

    # 7. Same content under different numbers
    by_hash = defaultdict(list)
    for rec in records:
        if (rec.get("content_hash") or "").strip():
            by_hash[rec["content_hash"]].append(rec)
    for members in by_hash.values():
        if len(members) > 1:
            ids = ", ".join(m["record_id"] for m in members)
            for rec in members:
                add(rec, "duplicate_content", "content_hash",
                    f"Identical content also filed as {ids}",
                    "Keep one master copy and archive or link the other")

    findings.sort(key=lambda f: (f["record_id"], f["check"]))
    return findings

In [ ]:
%%writefile cde/fix.py
"""Apply the safe, mechanical fixes and leave everything else for a person.

Only three kinds of problem are fixed automatically, because each has exactly one
correct answer that can be derived from the standard or from the record itself:
  * cosmetic file name problems (case, separators, missing revision)
  * retention class (derived from the document type)
  * classification of records flagged as containing personal data (raised to Restricted)
Everything else (duplicates, mismatches, dates, missing metadata ...) needs a human
decision and goes into the exceptions queue instead.
"""

from __future__ import annotations

from .standard import Standard

FIX_LOG_FIELDS = ["record_id", "check", "field", "old_value", "new_value", "reason"]


def apply_fixes(records: list[dict], findings: list[dict], std: Standard) -> tuple[list[dict], list[dict]]:
    fixed = [dict(r) for r in records]
    by_id = {r["record_id"]: r for r in fixed}
    log: list[dict] = []

    def record(rec_id, check, field, old, new, reason):
        log.append({"record_id": rec_id, "check": check, "field": field,
                    "old_value": old, "new_value": new, "reason": reason})

    for finding in findings:
        if finding["auto_fixable"] != "Y":
            continue
        rec = by_id[finding["record_id"]]
        check = finding["check"]

        if check == "filename_pattern":
            new_name = std.suggest_filename_fix(rec)
            if new_name and new_name != rec["file_name"]:
                record(rec["record_id"], check, "file_name", rec["file_name"], new_name,
                       "Normalised case/separators/revision to match metadata")
                rec["file_name"] = new_name

        elif check == "retention_class":
            expected = std.expected_retention(rec["doc_type"])
            if expected and rec["retention_class"] != expected:
                record(rec["record_id"], check, "retention_class", rec["retention_class"], expected,
                       f"Derived from document type {rec['doc_type']}")
                rec["retention_class"] = expected

        elif check == "personal_data_classification":
            if rec["classification"] != "Restricted":
                record(rec["record_id"], check, "classification", rec["classification"], "Restricted",
                       "Record contains personal data")
                rec["classification"] = "Restricted"

    return fixed, log

In [ ]:
%%writefile cde/report.py
"""Metrics, evaluation against the planted errors, charts and the markdown report."""

from __future__ import annotations

from collections import Counter, defaultdict
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402

from .audit import CHECKS  # noqa: E402

# Palette from the validated default set (slots 1 and 2, light mode).
SURFACE = "#fcfcfb"
INK = "#0b0b0b"
INK_SECONDARY = "#52514e"
MUTED = "#898781"
GRID = "#e1e0d9"
BLUE = "#2a78d6"
ORANGE = "#eb6834"

plt.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": ["DejaVu Sans"],
        "axes.facecolor": SURFACE,
        "figure.facecolor": SURFACE,
        "savefig.facecolor": SURFACE,
        "text.color": INK,
        "axes.labelcolor": INK_SECONDARY,
        "xtick.color": MUTED,
        "ytick.color": INK_SECONDARY,
    }
)


def _pct(part: int, whole: int) -> float:
    return round(100 * part / whole, 1) if whole else 0.0


def compute_metrics(records, findings_before, fix_log, findings_after) -> dict:
    total = len(records)
    before_ids = {f["record_id"] for f in findings_before}
    after_ids = {f["record_id"] for f in findings_after}
    return {
        "total_records": total,
        "findings_before": len(findings_before),
        "records_with_findings_before": len(before_ids),
        "record_error_rate_before_pct": _pct(len(before_ids), total),
        "findings_by_check_before": dict(Counter(f["check"] for f in findings_before)),
        "auto_fixes_applied": len(fix_log),
        "auto_fixes_by_check": dict(Counter(f["check"] for f in fix_log)),
        "findings_after": len(findings_after),
        "records_with_findings_after": len(after_ids),
        "record_error_rate_after_pct": _pct(len(after_ids), total),
        "findings_by_check_after": dict(Counter(f["check"] for f in findings_after)),
        "findings_by_severity_after": dict(Counter(f["severity"] for f in findings_after)),
    }


def evaluate_against_truth(findings: list[dict], truth: list[dict]) -> list[dict]:
    """Compare the audit's findings with the errors the generator planted."""
    found = {(f["record_id"], f["check"]) for f in findings}
    planted = {(t["record_id"], t["check"]) for t in truth}
    rows = []
    for check in CHECKS:
        p = {k for k in planted if k[1] == check}
        d = {k for k in found if k[1] == check}
        tp = len(p & d)
        fp = len(d - p)
        fn = len(p - d)
        rows.append(
            {
                "check": check,
                "planted": len(p),
                "detected": tp,
                "missed": fn,
                "false_positives": fp,
                "precision_pct": _pct(tp, tp + fp) if (tp + fp) else 100.0,
                "recall_pct": _pct(tp, tp + fn) if (tp + fn) else 100.0,
            }
        )
    return rows


# ------------------------------------------------------------------------------ charts
def _style_axes(ax):
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
    ax.spines["bottom"].set_color(GRID)
    ax.tick_params(length=0)
    ax.grid(axis="x", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)


def chart_findings_by_check(metrics: dict, path: Path) -> None:
    checks = sorted(CHECKS, key=lambda c: metrics["findings_by_check_before"].get(c, 0))
    before = [metrics["findings_by_check_before"].get(c, 0) for c in checks]
    after = [metrics["findings_by_check_after"].get(c, 0) for c in checks]
    labels = [c.replace("_", " ") for c in checks]
    ypos = range(len(checks))
    h = 0.34

    fig, ax = plt.subplots(figsize=(8.5, 5.2))
    ax.barh([y + h / 2 + 0.02 for y in ypos], before, height=h, color=BLUE, label="Before auto-fix")
    ax.barh([y - h / 2 - 0.02 for y in ypos], after, height=h, color=ORANGE, label="Left for a person")
    for y, b, a in zip(ypos, before, after):
        ax.text(b + 0.15, y + h / 2 + 0.02, str(b), va="center", fontsize=9, color=INK_SECONDARY)
        ax.text(a + 0.15, y - h / 2 - 0.02, str(a), va="center", fontsize=9, color=INK_SECONDARY)
    ax.set_yticks(list(ypos))
    ax.set_yticklabels(labels, fontsize=10)
    ax.set_xlabel("Findings")
    ax.set_title("Findings by check: before and after automatic fixes", loc="left", fontsize=12, color=INK)
    ax.legend(frameon=False, loc="lower right", fontsize=9, labelcolor=INK_SECONDARY)
    ax.set_xlim(0, max(before) + 2)
    _style_axes(ax)
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)


def chart_status(records: list[dict], path: Path) -> None:
    order = ["WIP", "SHARED", "PUBLISHED", "ARCHIVED"]
    counts = Counter(r["status"] for r in records)
    values = [counts.get(s, 0) for s in order]
    fig, ax = plt.subplots(figsize=(6.5, 3.8))
    ax.bar(order, values, width=0.5, color=BLUE)
    for x, v in zip(order, values):
        ax.text(x, v + 0.8, str(v), ha="center", fontsize=10, color=INK_SECONDARY)
    ax.set_title("Documents by CDE status", loc="left", fontsize=12, color=INK)
    ax.set_ylabel("Documents")
    ax.set_ylim(0, max(values) * 1.15)
    _style_axes(ax)
    ax.grid(axis="x", visible=False)
    ax.grid(axis="y", color=GRID, linewidth=0.8)
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)


def chart_rate_by_originator(records, findings_before, std, path: Path) -> None:
    # Records whose originator code is itself invalid are left out; they have no valid group.
    total = Counter(r["originator"] for r in records if r["originator"] in std.originators)
    flagged = defaultdict(set)
    rec_orig = {r["record_id"]: r["originator"] for r in records}
    for f in findings_before:
        if rec_orig[f["record_id"]] in total:
            flagged[rec_orig[f["record_id"]]].add(f["record_id"])
    codes = sorted(total, key=lambda c: _pct(len(flagged[c]), total[c]))
    rates = [_pct(len(flagged[c]), total[c]) for c in codes]
    labels = [f"{std.originators.get(c, c)} ({c})" for c in codes]
    fig, ax = plt.subplots(figsize=(7.5, 3.8))
    ax.barh(labels, rates, height=0.5, color=BLUE)
    for y, (r, c) in enumerate(zip(rates, codes)):
        ax.text(r + 0.5, y, f"{r:.0f}%  ({len(flagged[c])} of {total[c]})", va="center", fontsize=9, color=INK_SECONDARY)
    ax.set_xlim(0, max(rates) * 1.35)
    ax.set_xlabel("Share of that originator's records with at least one finding (before fixes)")
    ax.set_title("Records with findings by originator (errors planted at random)", loc="left", fontsize=12, color=INK)
    _style_axes(ax)
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)


# ------------------------------------------------------------------------------ markdown
def write_markdown(metrics: dict, evaluation: list[dict], exceptions: list[dict], as_of, path: Path) -> None:
    m = metrics
    lines = [
        "# Register audit report",
        "",
        f"Audit date: {as_of.isoformat()}. All data is synthetic.",
        "",
        "## Summary",
        "",
        f"- Records audited: **{m['total_records']}**",
        f"- Records with at least one finding before fixes: **{m['records_with_findings_before']}** "
        f"({m['record_error_rate_before_pct']}%)",
        f"- Automatic fixes applied: **{m['auto_fixes_applied']}**",
        f"- Records still needing a person after fixes: **{m['records_with_findings_after']}** "
        f"({m['record_error_rate_after_pct']}%)",
        "",
        "## Findings by check",
        "",
        "| Check | What it tests | Before | Fixed automatically | Left for a person |",
        "|---|---|---:|---:|---:|",
    ]
    for check, (_, description) in CHECKS.items():
        lines.append(
            f"| `{check}` | {description} | {m['findings_by_check_before'].get(check, 0)} "
            f"| {m['auto_fixes_by_check'].get(check, 0)} | {m['findings_by_check_after'].get(check, 0)} |"
        )
    lines += [
        "",
        "![Findings by check](figures/findings_by_check.png)",
        "",
        "## How well the audit finds known errors",
        "",
        "The generator plants a known set of errors and writes them to `data/ground_truth.csv`. "
        "The audit never reads that file; this table compares the two afterwards.",
        "",
        "| Check | Planted | Detected | Missed | False positives | Precision | Recall |",
        "|---|---:|---:|---:|---:|---:|---:|",
    ]
    for e in evaluation:
        lines.append(
            f"| `{e['check']}` | {e['planted']} | {e['detected']} | {e['missed']} | {e['false_positives']} "
            f"| {e['precision_pct']}% | {e['recall_pct']}% |"
        )
    lines += [
        "",
        "## Exceptions queue (needs a person)",
        "",
        f"{len(exceptions)} findings remain after the automatic fixes. "
        "The full list with suggested actions is in `reports/exceptions.csv`. First ten, highest severity first:",
        "",
        "| Record | File | Check | Severity | Suggested action |",
        "|---|---|---|---|---|",
    ]
    rank = {"High": 0, "Medium": 1, "Low": 2}
    for f in sorted(exceptions, key=lambda x: (rank[x["severity"]], x["record_id"]))[:10]:
        lines.append(
            f"| {f['record_id']} | `{f['file_name']}` | {f['check']} | {f['severity']} | {f['suggested_action']} |"
        )
    lines.append("")
    path.write_text("\n".join(lines), encoding="utf-8")

In [ ]:
%%writefile cde/pipeline.py
"""Run the whole workflow: generate -> audit -> fix -> re-audit -> report.

    python -m cde.pipeline                 # defaults: seed 42, 120 documents, as-of 2026-11-12
    python -m cde.pipeline --seed 7 --documents 200
"""

from __future__ import annotations

import argparse
import json
from datetime import date
from pathlib import Path

from . import report
from .audit import FINDING_FIELDS, audit
from .fix import FIX_LOG_FIELDS, apply_fixes
from .generate import generate, write_register, write_rows
from .standard import ROOT, Standard


def run(out: Path = ROOT, seed: int = 42, n_documents: int = 120, as_of: date = date(2026, 11, 12)) -> dict:
    std = Standard.load()
    data_dir, reports_dir = out / "data", out / "reports"
    figures_dir = reports_dir / "figures"
    figures_dir.mkdir(parents=True, exist_ok=True)

    # 1. Synthetic register with planted errors (plus the answer key).
    records, truth = generate(std, seed=seed, n_documents=n_documents, as_of=as_of)
    write_register(records, data_dir / "register_raw.csv")
    write_rows(truth, data_dir / "ground_truth.csv", ["record_id", "check", "detail"])

    # 2. Audit the raw register.
    findings_before = audit(records, std, as_of)
    write_rows(findings_before, reports_dir / "findings_before.csv", FINDING_FIELDS)

    # 3. Apply the safe fixes and log every change.
    cleaned, fix_log = apply_fixes(records, findings_before, std)
    write_register(cleaned, data_dir / "register_clean.csv")
    write_rows(fix_log, reports_dir / "fix_log.csv", FIX_LOG_FIELDS)

    # 4. Re-audit: whatever is left goes to a person.
    findings_after = audit(cleaned, std, as_of)
    write_rows(findings_after, reports_dir / "exceptions.csv", FINDING_FIELDS)

    # 5. Metrics, evaluation against the planted errors, charts, report.
    metrics = report.compute_metrics(records, findings_before, fix_log, findings_after)
    evaluation = report.evaluate_against_truth(findings_before, truth)
    metrics["evaluation"] = evaluation
    (reports_dir / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
    write_rows(
        evaluation,
        reports_dir / "audit_evaluation.csv",
        ["check", "planted", "detected", "missed", "false_positives", "precision_pct", "recall_pct"],
    )
    report.chart_findings_by_check(metrics, figures_dir / "findings_by_check.png")
    report.chart_status(records, figures_dir / "records_by_status.png")
    report.chart_rate_by_originator(records, findings_before, std, figures_dir / "finding_rate_by_originator.png")
    report.write_markdown(metrics, evaluation, findings_after, as_of, reports_dir / "audit_report.md")
    return metrics


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--documents", type=int, default=120)
    parser.add_argument("--as-of", type=date.fromisoformat, default=date(2026, 11, 12))
    parser.add_argument("--out", type=Path, default=ROOT)
    args = parser.parse_args()
    m = run(args.out, args.seed, args.documents, args.as_of)
    print(
        f"{m['total_records']} records | before: {m['records_with_findings_before']} flagged "
        f"({m['record_error_rate_before_pct']}%) | auto-fixes: {m['auto_fixes_applied']} | "
        f"after: {m['records_with_findings_after']} flagged ({m['record_error_rate_after_pct']}%)"
    )


if __name__ == "__main__":
    main()

In [ ]:
%%writefile tests/test_standard.py
import pytest

from cde.standard import Standard


@pytest.fixture(scope="module")
def std():
    return Standard.load()


def test_valid_filename_parses(std):
    parsed = std.parse_filename("MVT-ARC-S3-DR-CV-0012_P02.pdf")
    assert parsed == {
        "originator": "ARC", "section": "S3", "doc_type": "DR",
        "discipline": "CV", "number": "0012", "revision": "P02", "ext": "pdf",
    }


@pytest.mark.parametrize(
    "name",
    [
        "mvt-arc-s3-dr-cv-0012_p02.pdf",      # lower case
        "MVT_ARC_S3_DR_CV_0012_P02.pdf",      # underscores as separators
        "MVT-ARC-S3-DR-CV-0012.pdf",          # no revision
        "MVT-ARC-S3-DR-CV-0012_P02 FINAL.pdf",  # extra word
        "MVT-ARC-S3-DR-CV-12_P02.pdf",        # number not 4 digits
        "XXX-ARC-S3-DR-CV-0012_P02.pdf",      # wrong project code
        "",
    ],
)
def test_bad_filenames_rejected(std, name):
    assert std.parse_filename(name) is None


def test_build_and_parse_round_trip(std):
    name = std.build_filename("KTL", "ZZ", "MS", "HS", "0101", "C01", "docx")
    assert std.parse_filename(name)["originator"] == "KTL"
    assert name == "MVT-KTL-ZZ-MS-HS-0101_C01.docx"


def _rec(file_name):
    return {
        "file_name": file_name, "originator": "ARC", "section": "S3", "doc_type": "DR",
        "discipline": "CV", "number": "0012", "revision": "P02",
    }


def test_cosmetic_problems_get_an_automatic_fix(std):
    expected = "MVT-ARC-S3-DR-CV-0012_P02.pdf"
    assert std.suggest_filename_fix(_rec("mvt-arc-s3-dr-cv-0012_p02.pdf")) == expected
    assert std.suggest_filename_fix(_rec("MVT_ARC_S3_DR_CV_0012_P02.pdf")) == expected
    assert std.suggest_filename_fix(_rec("MVT-ARC-S3-DR-CV-0012.pdf")) == expected


def test_non_cosmetic_problems_are_left_for_a_person(std):
    assert std.suggest_filename_fix(_rec("MVT-ARC-S3-DR-CV-0012_P02 FINAL.pdf")) is None
    assert std.suggest_filename_fix(_rec("MVT-ARC-S3-DR-CV-0099_P02.pdf")) is None


def test_retention_lookup(std):
    assert std.expected_retention("DR") == "R12"
    assert std.expected_retention("LT") == "R6"
    assert std.expected_retention("ZZ") is None

In [ ]:
%%writefile tests/test_pipeline.py
from datetime import date

import pytest

from cde.audit import CHECKS, audit
from cde.fix import apply_fixes
from cde.generate import generate
from cde.report import evaluate_against_truth
from cde.standard import Standard

AS_OF = date(2026, 11, 12)


@pytest.fixture(scope="module")
def std():
    return Standard.load()


def test_clean_register_has_no_findings(std):
    """No planted errors means no findings: the checks do not raise false alarms."""
    records, truth = generate(std, seed=1, n_documents=150, as_of=AS_OF, inject_errors=False)
    assert truth == []
    assert audit(records, std, AS_OF) == []


@pytest.mark.parametrize("seed", [1, 2, 3, 42, 99])
def test_every_planted_error_is_found_and_nothing_extra(std, seed):
    records, truth = generate(std, seed=seed, n_documents=120, as_of=AS_OF)
    findings = audit(records, std, AS_OF)
    for row in evaluate_against_truth(findings, truth):
        assert row["missed"] == 0, row
        assert row["false_positives"] == 0, row


def test_every_check_is_exercised_by_the_generator(std):
    records, truth = generate(std, seed=42, n_documents=120, as_of=AS_OF)
    assert {t["check"] for t in truth} == set(CHECKS)


def test_fixes_remove_only_auto_fixable_findings(std):
    records, _ = generate(std, seed=42, n_documents=120, as_of=AS_OF)
    before = audit(records, std, AS_OF)
    cleaned, log = apply_fixes(records, before, std)
    after = audit(cleaned, std, AS_OF)

    fixable = {(f["record_id"], f["check"]) for f in before if f["auto_fixable"] == "Y"}
    remaining = {(f["record_id"], f["check"]) for f in after}
    before_keys = {(f["record_id"], f["check"]) for f in before}

    assert not (fixable & remaining) or all(k[1] == "filename_pattern" for k in fixable & remaining)
    assert remaining <= before_keys            # fixing never creates a new finding
    assert len(log) == len(before_keys - remaining)


def test_fixing_does_not_change_the_input(std):
    records, _ = generate(std, seed=42, n_documents=120, as_of=AS_OF)
    snapshot = [dict(r) for r in records]
    apply_fixes(records, audit(records, std, AS_OF), std)
    assert records == snapshot


def test_same_seed_gives_identical_output(std):
    a, ta = generate(std, seed=7, n_documents=80, as_of=AS_OF)
    b, tb = generate(std, seed=7, n_documents=80, as_of=AS_OF)
    assert a == b and ta == tb


def test_synthetic_emails_use_reserved_test_domain(std):
    records, _ = generate(std, seed=42, n_documents=120, as_of=AS_OF)
    assert all(r["author_email"].endswith("@example.test") for r in records if r["author_email"])

In [ ]:
%%writefile docs/01-naming-and-metadata-standard.md
# 01. Naming and metadata standard

Project: Marlow Valley Tunnel Scheme (a fictional project used for this demonstration).
Machine-readable version: [`config/standard.json`](../config/standard.json). The audit code reads that file, so
changing a code list there changes what the checks accept.

This standard is modelled on the ideas in ISO 19650 (information management using building information
modelling): a consistent file name built from coded fields, a status and suitability for every document, and
a revision scheme that separates work in progress from published information. It is simplified for the demo
and is not a copy of any organisation's standard.

## 1. File name

```
MVT-ARC-S3-DR-CV-0012_P02.pdf
|   |   |  |  |  |    |   |
|   |   |  |  |  |    |   +-- file extension (lower case)
|   |   |  |  |  |    +------ revision (P01..P99 or C01..C99)
|   |   |  |  |  +----------- document number (4 digits)
|   |   |  |  +-------------- discipline code
|   |   |  +----------------- document type code
|   |   +-------------------- section code
|   +------------------------ originator code
+---------------------------- project code (always MVT)
```

Rules:

- Fields are separated by hyphens. The revision is separated from the number by one underscore.
- Codes are upper case. The extension is lower case.
- No spaces and no extra words (for example `FINAL` or `copy`). Status belongs in metadata, not in the name.
- The document number is unique within the same originator, section, type and discipline.

Regular expression used by the audit:

```
^MVT-[A-Z]{3}-[A-Z0-9]{2}-[A-Z]{2}-[A-Z]{2}-\d{4}_[PC]\d{2}\.[a-z0-9]{2,4}$
```

## 2. Code lists

**Originator**

| Code | Organisation (fictional) |
|---|---|
| ARC | Arden Civil Design |
| KTL | Kestrel Tunnelling |
| FGS | Fieldgate Surveys |
| MVC | Marlow Valley Constructors |
| CLT | Client team |

**Section**

| Code | Meaning |
|---|---|
| S1 to S6 | Tunnel sections 1 to 6 |
| ZZ | Project-wide |

**Document type**

| Code | Meaning | Code | Meaning |
|---|---|---|---|
| DR | Drawing | CA | Calculation |
| RP | Report | LT | Letter |
| MS | Method statement | MI | Meeting minutes |
| SP | Specification | RA | Risk assessment |

**Discipline**

| Code | Meaning | Code | Meaning |
|---|---|---|---|
| CV | Civil | PM | Project management |
| ST | Structural | SU | Survey |
| GE | Geotechnical | EN | Environmental |
| ME | Mechanical | HS | Health and safety |
| EL | Electrical | | |

New codes are added only by the Information Manager, in `config/standard.json`, with a note in the change log.

## 3. Status, suitability and revision

Every document has a CDE status, a suitability code and a revision. They must agree.

| Status | Meaning | Allowed suitability | Revision prefix |
|---|---|---|---|
| WIP | Being worked on by its author, not visible to others | S0 | P |
| SHARED | Visible to the project team for coordination or comment | S1, S2, S3 | P |
| PUBLISHED | Approved and issued | S4, A | C |
| ARCHIVED | Superseded or closed, kept for the record | any | any |

| Suitability | Meaning |
|---|---|
| S0 | Work in progress (not for sharing) |
| S1 | Suitable for coordination |
| S2 | Suitable for information |
| S3 | Suitable for review and comment |
| S4 | Suitable for stage approval |
| A | Authorised and accepted |

Revisions: `P01, P02 ...` are preliminary (before approval). `C01, C02 ...` are contractual (approved).

## 4. Register metadata

Every document has one row in the register. All fields are required except `review_due` on ARCHIVED documents.

| Field | Content |
|---|---|
| `record_id` | Unique register row ID (`REG-0001`) |
| `file_name` | As in section 1 |
| `title` | Human-readable title |
| `originator`, `section`, `doc_type`, `discipline`, `number`, `revision` | Must match the file name exactly |
| `status`, `suitability` | As in section 3 |
| `author`, `author_email` | Person responsible (personal data, see doc 03) |
| `created`, `last_modified`, `review_due` | ISO dates (`YYYY-MM-DD`); `last_modified` is never before `created` |
| `content_hash` | Fingerprint of the file content, used to find identical copies |
| `contains_personal_data` | `Y` or `N` |
| `classification` | `Internal` or `Restricted`. Anything with `contains_personal_data = Y` is `Restricted` |
| `retention_class` | `R12` or `R6`, set by document type (doc 03) |

## 5. What the audit checks

The register audit (`cde/audit.py`) tests each of these rules.

| Check | Rule |
|---|---|
| `missing_metadata` | Required fields are filled in |
| `filename_pattern` | File name matches the pattern in section 1 |
| `filename_metadata_mismatch` | Codes and revision in the file name equal the register fields |
| `unknown_code` | Originator, section, type, discipline, status, suitability and revision format are valid |
| `status_consistency` | Suitability and revision prefix are allowed for the status |
| `duplicate_number` | No two records share originator, section, type, discipline, number and revision |
| `duplicate_content` | No two records share a content hash |
| `overdue_review` | Non-archived documents are not past their review date |
| `date_error` | Dates are valid, `last_modified` is not before `created`, `created` is not in the future |
| `personal_data_classification` | Personal data is classed Restricted |
| `retention_class` | Retention class matches the document type |

In [ ]:
%%writefile docs/02-information-management-procedure.md
# 02. Information management procedure

This procedure describes how information moves through the common data environment (CDE) on the fictional
Marlow Valley Tunnel Scheme. It is written so that a new team member can follow it without having to ask.

## 1. Roles

| Role | Responsibility |
|---|---|
| **Author** (originator's staff) | Names files correctly, completes the metadata, moves documents through the states, reviews their own documents by the review date |
| **Document controller** | Runs the weekly register audit, applies the safe fixes, chases the exceptions queue, keeps the transmittal log |
| **Information Manager** | Owns the standard and code lists, decides on new codes, resolves duplicate numbers, reports on information quality, trains users |
| **Project manager** | Receives the monthly quality report and escalates repeated problems with an originator |

## 2. Document lifecycle

```
 WIP  ->  SHARED  ->  PUBLISHED  ->  ARCHIVED
 (S0)    (S1-S3)      (S4, A)
 P rev    P rev        C rev
```

1. **WIP.** The author works in their own area. Nobody else can see the file. Revision `P01`, suitability `S0`.
2. **SHARED.** The author moves the file to the shared area for coordination or comment. Suitability `S1`, `S2`
   or `S3`. Revision stays `P` and increases each time a new version is shared.
3. **PUBLISHED.** After approval the document is issued. Suitability `S4` or `A`. Revision changes to `C01`
   and then increases with each approved change.
4. **ARCHIVED.** When superseded or when the work is closed, the document is archived and kept according to
   its retention class. Archived documents are read-only.

A published document is never edited in place. A change is a new revision.

## 3. Before a file goes into the CDE

The author checks the following. The upload check in the CDE (or, in this demo, the register audit) repeats it.

1. The file name follows the pattern in doc 01 and uses only approved codes.
2. The document number has not been used before for the same originator, section, type and discipline.
3. The register entry is complete and its codes match the file name.
4. Status, suitability and revision agree with each other.
5. If the file contains personal data, `contains_personal_data` is `Y` and the file is classed `Restricted`.
6. A review date is set (not required for archived documents).

## 4. Weekly register audit

Run by the document controller every week (and before any contractual submission).

1. Export the register and run the audit (`python -m cde.pipeline`, or the equivalent check in the live system).
2. Apply the automatic fixes. Each one is written to `fix_log.csv` with the old and new value, so any fix can
   be reversed.
3. Review the exceptions queue (`exceptions.csv`). Every item has a severity and a suggested action.
4. Send each exception to the person who can resolve it, with the target date from the table below.
5. Re-run the audit after the fixes. The count of open exceptions is the headline figure in the monthly report.

| Severity | Examples | Target to resolve |
|---|---|---|
| High | Duplicate number, file name and metadata disagree, personal data not Restricted, status inconsistent | 2 working days |
| Medium | Missing metadata, unknown code, overdue review, duplicate content | 5 working days |
| Low | Retention class wrong, date order wrong | 10 working days |

What counts as a safe automatic fix: only changes with exactly one correct answer that can be derived from the
standard (file name case and separators, retention class from document type, raising classification to
Restricted when personal data is flagged). Anything that needs a judgement, such as deciding which of two
duplicates is the real one, goes to a person.

## 5. Access

| Group | WIP | SHARED | PUBLISHED | ARCHIVED | Restricted documents |
|---|---|---|---|---|---|
| Authors | Own files only | Read and write own, read others | Read | Read | Only if named on the document |
| Project team | None | Read | Read | Read | No |
| Document controllers | Metadata only | Read and write metadata | Read and write metadata | Read | Metadata only, content by exception |
| Information Manager | Metadata only | Read and write | Read | Read and restore | Yes, for audit and data requests |
| Client team | None | By agreement | Read | None | No |

Access is removed on the day a person leaves the project, and the list is reviewed every quarter.

## 6. Transmittals

Every formal issue of published documents to another organisation is recorded in the transmittal log:
transmittal number, date, sender, recipient, the document numbers and revisions issued, and the reason for
issue. The log lets anyone answer "which revision did they have on that date?".

## 7. Training and onboarding

New starters get a 30-minute session covering the naming standard, the lifecycle above and how to read the
audit report, and a one-page quick reference of the file name pattern. The Information Manager reviews the
monthly report for repeat patterns by originator and offers a short refresher where the same error keeps
appearing.

## 8. Reporting

Monthly report to the project manager:

- Records audited and share with at least one finding, before and after fixes
- Findings by check and by originator
- Open exceptions by severity and age
- Actions taken (new codes, refreshers, access changes)

In [ ]:
%%writefile docs/03-data-protection-and-retention.md
# 03. Data protection and retention note

This note explains how the register and the CDE handle personal data on the fictional Marlow Valley Tunnel
Scheme. It is a worked example for a portfolio. It is not legal advice, and the retention periods are
illustrative: on a real project they come from the contract, the organisation's retention policy and the
advice of its data protection officer.

All people and e-mail addresses in this repository are invented. E-mail addresses use the reserved
`example.test` domain.

## 1. Legislation

In the UK, personal data is governed by the UK GDPR and the Data Protection Act 2018. The points below apply
those rules to a project information register.

## 2. What personal data the register holds

| Data | Where | Why it is needed |
|---|---|---|
| Author name | `author` | Accountability: who is responsible for the document |
| Author work e-mail | `author_email` | To contact the author about the document |
| Names inside documents (letters, minutes, some risk assessments) | The documents themselves | Content of the record; flagged with `contains_personal_data = Y` |

The register holds no home addresses, dates of birth, ID numbers, health data or other special category data.
If a document does contain such data (for example an accident report), it is classed Restricted and handled
under the organisation's specific procedure.

## 3. Principles applied

| Principle | How the demo applies it |
|---|---|
| Lawfulness, fairness, transparency | Project staff are told the register records their name and work e-mail and why |
| Purpose limitation | Names and e-mails are used to run the project's information management, not for other purposes |
| Data minimisation | Only name and work e-mail are held per author; no personal contact details |
| Accuracy | The weekly audit checks owner and e-mail fields are present, and people are asked to correct them |
| Storage limitation | Retention classes below, applied by document type |
| Integrity and confidentiality | Documents with personal data are classed Restricted, with access limited to named people (doc 02, section 5) |
| Accountability | Fixes are logged with old and new values; the monthly report records actions taken |

## 4. How classification works

- Every register row has `contains_personal_data` (`Y` or `N`).
- If it is `Y`, `classification` must be `Restricted`. The audit raises a High severity finding if it is not, and
  the automatic fix sets it to Restricted. Raising the level is the safe direction, so it can be done without
  asking the author.
- Letters and meeting minutes are always flagged `Y`, because they name people.

## 5. Retention schedule (illustrative)

| Class | Applies to | Period | Notes |
|---|---|---|---|
| R12 | Drawings, reports, method statements, specifications, risk assessments, calculations | 12 years after project completion | Design and construction records are often kept for the length of the contractual liability period; many contracts use 6 or 12 years |
| R6 | Letters, meeting minutes | 6 years after project completion | Correspondence kept for the typical limitation period for contract claims |

The class is set from the document type. The audit raises a Low severity finding if the class does not match,
and the automatic fix applies the correct one. Where a document is kept beyond the point it would otherwise be
deleted (for example a legal hold), the reason and the approver are recorded.

At the end of the retention period, documents are reviewed for deletion. Personal data inside them is deleted
or anonymised unless there is a reason to keep it.

## 6. Individuals' rights and requests

If someone asks what personal data is held about them (a subject access request) or asks for correction or
deletion:

1. Pass the request to the data protection contact straight away. The legal deadline is generally one month,
   so it must not wait.
2. The Information Manager searches the register by `author` and `author_email`, and searches Restricted
   documents by name.
3. The data protection contact decides what is disclosed, corrected or deleted. The Information Manager carries
   out the technical steps and records them.

## 7. If personal data is shared or exposed by mistake

1. Stop the sharing (move the document back to WIP or Restricted, remove the link).
2. Tell the data protection contact immediately.
3. Record what happened, which documents and which people were involved, and who saw it.
4. The data protection contact decides whether to report to the Information Commissioner's Office. If a report
   is required it must normally be made within 72 hours of becoming aware of the breach.
5. Add the cause to the lessons learned (for example, a missing classification) and adjust the audit check if
   it would have caught the problem.

## 8. Limits of this demo

- The data is synthetic, so no real person is affected.
- A real CDE would enforce access controls in the system itself. The demo only checks that the classification
  field is right.
- Retention periods and the breach process must be confirmed against the real contract and policy before use.

In [ ]:
%%writefile requirements.txt
matplotlib>=3.8
pytest>=8.0

In [ ]:
%%writefile .gitignore
__pycache__/
*.pyc
.pytest_cache/
.venv/

In [ ]:
%%writefile README.md
# CDE information management demo

A small, runnable example of how information is organised, checked and reported on a large infrastructure
project. It uses a fictional tunnelling scheme (the Marlow Valley Tunnel Scheme) and entirely synthetic data.

It covers four things an information management team does every week:

1. **A standard**: how files are named, what metadata each document carries, and how status, suitability and
   revision relate (modelled on ISO 19650 ideas, simplified).
2. **A register audit**: 11 automated checks that find naming, metadata, duplication, review and data
   protection problems in a document register.
3. **Safe automatic fixes plus an exceptions queue**: mechanical problems are corrected and logged; anything
   that needs a judgement is routed to a person with a suggested action.
4. **Reporting**: a report and charts showing the state of the register before and after.

## Results from the included run

Seed 42, 126 register records (120 documents plus 6 deliberate duplicate copies), audit date 2026-11-12.

| Measure | Result |
|---|---|
| Records with at least one finding, before fixes | 51 of 126 (40.5%) |
| Fixed automatically and logged | 12 findings (file name format 4, retention class 4, personal data classification 4) |
| Records still needing a person | 39 of 126 (31.0%) |
| Planted errors found by the audit | 51 of 51, with 0 false positives |

![Findings by check](reports/figures/findings_by_check.png)

Open `reports/audit_report.md` for the full tables, `reports/exceptions.csv` for the queue with suggested
actions, and `reports/fix_log.csv` for every change made.

### How to read these numbers

- The register is synthetic and the errors in it were planted by the generator, at rates I chose. The 40.5%
  is not a typical real-world error rate. It is just the size of the test.
- The audit finds all 51 planted errors with no false alarms. That shows the checks do what they were written
  to do and do not complain about clean records (a test confirms this on error-free data). It does not show how
  they would perform on a real register, where errors are messier and the rules would need local tuning.
- Only 12 of the 51 problems are fixed automatically, on purpose. The other 39 need a decision, for example
  which of two documents with the same number is the real one. The aim is a short, clear list for a person, not
  a register that quietly rewrites itself.

## Quick start

Requires Python 3.10 or later.

```bash
git clone <your-repo-url>
cd cde-information-management-demo
pip install -r requirements.txt
python -m cde.pipeline        # regenerates data/ and reports/
python -m pytest              # runs the tests
```

Options: `python -m cde.pipeline --seed 7 --documents 200 --as-of 2026-12-01`. The same seed always gives
the same output.

## How it works

```
generate.py   builds a synthetic register and plants known errors (answer key: data/ground_truth.csv)
     |
audit.py      runs the 11 checks on the register (it never reads the answer key)
     |
fix.py        applies the three safe fixes, writes reports/fix_log.csv
     |
audit.py      re-runs on the fixed register -> reports/exceptions.csv
     |
report.py     metrics, comparison with the answer key, charts, reports/audit_report.md
```

The checks:

| Check | Tests |
|---|---|
| `filename_pattern` | File name matches the standard |
| `filename_metadata_mismatch` | Codes and revision in the name equal the register fields |
| `unknown_code` | Codes are on the approved lists |
| `missing_metadata` | Required fields are filled in |
| `status_consistency` | Status, suitability and revision agree |
| `duplicate_number` | No two records share a number and revision |
| `duplicate_content` | No identical content under two numbers |
| `overdue_review` | Review date not passed |
| `date_error` | Dates valid and in order |
| `personal_data_classification` | Personal data is classed Restricted |
| `retention_class` | Retention class matches document type |

## Repository contents

```
config/standard.json                      code lists and rules the audit uses
docs/01-naming-and-metadata-standard.md   the standard
docs/02-information-management-procedure.md  roles, lifecycle, weekly audit, access, reporting
docs/03-data-protection-and-retention.md  personal data handling and retention schedule
cde/                                      standard, generator, audit, fix, report, pipeline
tests/                                    23 tests
data/                                     register_raw.csv, register_clean.csv, ground_truth.csv
reports/                                  findings, fix log, exceptions, metrics, report, figures
```

## Design decisions

- **Rules live in one config file.** Adding a section or document type is a one-line change to
  `config/standard.json`, not a code change.
- **Every automatic change is logged with the old and new value**, so it can be checked or reversed.
- **Only changes with one correct answer are automated.** Raising a classification to Restricted is safe;
  guessing which duplicate to keep is not.
- **Measured against a known answer.** Planting the errors myself lets me report missed errors and false
  alarms instead of just saying the checks work.

## Limitations and what a live project would change

- The data is synthetic and the ground truth comes from the same author as the checks, so the 100% result is
  a consistency check, not an accuracy claim.
- The content hash is a stand-in. On a live project it would be calculated from the files in the CDE.
- Access control is documented (doc 02) but not enforced here. A real CDE enforces it in the system.
- The retention periods are illustrative and must come from the contract and organisation policy.
- The register would be pulled from the CDE through its export or API rather than a CSV, and the audit would
  run on a schedule, with the exceptions sent to the right people automatically.
- Next steps: check file names at upload time, add a transmittal log check, and trend the findings over time.

## Data note

All names, organisations, e-mail addresses and documents are invented. E-mail addresses use the reserved
`example.test` domain. The project is fictional and is not based on any real scheme, client or contractor.

## Step 3. Run the whole workflow

Generate the register, audit it, apply the safe fixes, re-audit and build the report.

In [ ]:
!python -m cde.pipeline

## Step 4. Results

In [ ]:
import json
import pandas as pd
from IPython.display import Image, Markdown, display

m = json.load(open("reports/metrics.json"))
display(Markdown(f"""
**Records audited:** {m['total_records']}
**Records with at least one finding before fixes:** {m['records_with_findings_before']} ({m['record_error_rate_before_pct']}%)
**Fixed automatically and logged:** {m['auto_fixes_applied']}
**Records still needing a person:** {m['records_with_findings_after']} ({m['record_error_rate_after_pct']}%)
"""))
display(Image("reports/figures/findings_by_check.png"))

### How well did the audit find the planted errors?

The generator writes the errors it planted to `data/ground_truth.csv`. The audit never reads that file; this
table compares the two afterwards. A perfect score here shows the checks do what they were written to do on
synthetic data. It is **not** a claim about accuracy on a real register.

In [ ]:
display(pd.read_csv("reports/audit_evaluation.csv"))

### The dataset: the raw register (with errors)

In [ ]:
raw = pd.read_csv("data/register_raw.csv", dtype=str, keep_default_na=False)
print(f"{len(raw)} rows, {len(raw.columns)} columns")
display(raw.head(15))

### Everything that was found, and what happened to it

In [ ]:
print("All findings before fixes")
display(pd.read_csv("reports/findings_before.csv").head(20))
print("Automatic fixes applied (every change is logged with old and new value)")
display(pd.read_csv("reports/fix_log.csv"))

### Exceptions queue: what still needs a person

In [ ]:
exceptions = pd.read_csv("reports/exceptions.csv")
rank = {"High": 0, "Medium": 1, "Low": 2}
exceptions = exceptions.sort_values(by=["severity", "record_id"], key=lambda s: s.map(rank) if s.name == "severity" else s)
print(f"{len(exceptions)} open exceptions")
display(exceptions[["record_id", "file_name", "check", "severity", "suggested_action"]])

### More charts

In [ ]:
display(Image("reports/figures/records_by_status.png"))
display(Image("reports/figures/finding_rate_by_originator.png"))

## Step 5. Run the tests

In [ ]:
!python -m pytest -q

## Step 6. Try your own run

Change the seed, the number of documents or the audit date and run again. The same seed always gives the same
output. This run is saved to `/content/my_run`, so it does not overwrite the results above (open the folder icon on the
left to browse the CSVs and charts).

In [ ]:
!python -m cde.pipeline --seed 7 --documents 200 --as-of 2026-12-01 --out /content/my_run

## Step 7. Download everything

In [ ]:
!cd /content && rm -f cde-information-management-demo.zip && zip -rq cde-information-management-demo.zip cde-information-management-demo -x "*/__pycache__/*" "*/.pytest_cache/*"
try:
    from google.colab import files
    files.download("/content/cde-information-management-demo.zip")
except ImportError:
    print("Zip saved at /content/cde-information-management-demo.zip")